In [9]:
import os, torch, random
import torchvision
import torchvision.transforms as transforms
from torch import nn
from torch.utils.data import DataLoader
import numpy as np

import zipfile
import gdown

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

BASE_DIR = r"D:\pycharm\CNN"
DATA_DIR = os.path.join(BASE_DIR, "data", "food-11")
MODEL_DIR = os.path.join(BASE_DIR, "models")
ZIP_PATH = os.path.join(BASE_DIR, "food-11.zip")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset...")
    url = "https://drive.google.com/uc?id=1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy"
    gdown.download(url, ZIP_PATH, quiet=False)
else:
    print("Dataset zip already exists, skip download.")

train_path = os.path.join(DATA_DIR, "training")
if not os.path.exists(train_path):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
else:
    print("Dataset already extracted.")

assert os.path.exists(DATA_DIR), "DATA_DIR 不存在"
print("DATA_DIR ready:", DATA_DIR)
print("MODEL_DIR ready:", MODEL_DIR)


seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

Using device: cuda
Dataset zip already exists, skip download.
Dataset already extracted.
DATA_DIR ready: D:\pycharm\CNN\data\food-11
MODEL_DIR ready: D:\pycharm\CNN\models


In [10]:
train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                     [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                     [0.229, 0.224, 0.225])
])

train_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "training", "labeled"), transform=train_tf)
val_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "validation"), transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

In [11]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256), nn.ReLU(),
            nn.Linear(256, 11)
        )

    def forward(self, x): return self.net(x)


model = CNN().to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def evaluate():
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total


best = 0
for epoch in range(10):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad();
        loss.backward();
        optimizer.step()

    acc = evaluate()
    print(f"Epoch {epoch + 1}: {acc:.2f}%")

    if acc > best:
        best = acc
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "level1.pth"))

print("Best:", best)

Epoch 1: 23.79%
Epoch 2: 31.97%
Epoch 3: 36.06%
Epoch 4: 37.73%
Epoch 5: 35.45%
Epoch 6: 35.00%
Epoch 7: 38.94%
Epoch 8: 36.97%
Epoch 9: 40.30%
Epoch 10: 37.73%
Best: 40.303030303030305
